# ENV SETUP

In [21]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/credit-card-fraud/card_transdata.csv


# IMPORT DATA

In [22]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "card_transdata.csv"

# Load the latest version
df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "dhanushnarayananr/credit-card-fraud",
  file_path,
  # Provide any additional arguments like 
  # sql_query or pandas_kwargs. See the 
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

print(df.head())

   distance_from_home  distance_from_last_transaction  \
0           57.877857                        0.311140   
1           10.829943                        0.175592   
2            5.091079                        0.805153   
3            2.247564                        5.600044   
4           44.190936                        0.566486   

   ratio_to_median_purchase_price  repeat_retailer  used_chip  \
0                        1.945940              1.0        1.0   
1                        1.294219              1.0        0.0   
2                        0.427715              1.0        0.0   
3                        0.362663              1.0        1.0   
4                        2.222767              1.0        1.0   

   used_pin_number  online_order  fraud  
0              0.0           0.0    0.0  
1              0.0           0.0    0.0  
2              0.0           1.0    0.0  
3              0.0           1.0    0.0  
4              0.0           1.0    0.0  


# DATA SANITY CHECK

In [23]:
# get the dataset information
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 8 columns):
 #   Column                          Non-Null Count    Dtype  
---  ------                          --------------    -----  
 0   distance_from_home              1000000 non-null  float64
 1   distance_from_last_transaction  1000000 non-null  float64
 2   ratio_to_median_purchase_price  1000000 non-null  float64
 3   repeat_retailer                 1000000 non-null  float64
 4   used_chip                       1000000 non-null  float64
 5   used_pin_number                 1000000 non-null  float64
 6   online_order                    1000000 non-null  float64
 7   fraud                           1000000 non-null  float64
dtypes: float64(8)
memory usage: 61.0 MB


In [24]:
# numeric variables statistics
df.describe()

,distance_from_home,distance_from_last_transaction,ratio_to_median_purchase_price,repeat_retailer,used_chip,used_pin_number,online_order,fraud
count,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000
mean,26.628792,5.036519,1.824182,0.881536,0.350399,0.100608,0.650552,0.087403
std,65.390784,25.843093,2.799589,0.323157,0.477095,0.300809,0.476796,0.282425
min,0.004874,0.000118,0.004399,0.000000,0.000000,0.000000,0.000000,0.000000
25%,3.878008,0.296671,0.475673,1.000000,0.000000,0.000000,0.000000,0.000000
50%,9.967760,0.998650,0.997717,1.000000,0.000000,0.000000,1.000000,0.000000
75%,25.743985,3.355748,2.096370,1.000000,1.000000,0.000000,1.000000,0.000000
max,10632.723672,11851.104565,267.802942,1.000000,1.000000,1.000000,1.000000,1.000000


In [25]:
df.isnull().sum()
# no missing value

distance_from_home                0
distance_from_last_transaction    0
ratio_to_median_purchase_price    0
repeat_retailer                   0
used_chip                         0
used_pin_number                   0
online_order                      0
fraud                             0
dtype: int64

In [26]:
# change binary variables data type from float to int
binary_col = ['repeat_retailer','used_chip','used_pin_number','online_order','fraud']
for col in binary_col:
    df[col] = df[col].astype(int)
df.dtypes

distance_from_home                float64
distance_from_last_transaction    float64
ratio_to_median_purchase_price    float64
repeat_retailer                     int64
used_chip                           int64
used_pin_number                     int64
online_order                        int64
fraud                               int64
dtype: object

In [27]:
# check unique value distributions % for those binary variables
for col in binary_col:
    print(df[col].value_counts(normalize=True))

repeat_retailer
1    0.881536
0    0.118464
Name: proportion, dtype: float64
used_chip
0    0.649601
1    0.350399
Name: proportion, dtype: float64
used_pin_number
0    0.899392
1    0.100608
Name: proportion, dtype: float64
online_order
1    0.650552
0    0.349448
Name: proportion, dtype: float64
fraud
0    0.912597
1    0.087403
Name: proportion, dtype: float64


# DATA PRE-PROCESSING

In [28]:
# train/test data split
from sklearn.model_selection import train_test_split
y = df['fraud']
x = df.drop(columns=['fraud'])
x_train, x_test, y_train, y_test = train_test_split(x,y, 
                                                    test_size=0.2, 
                                                    stratify=y, #Keeps the fraud rate the same in train and test
                                                    random_state=84)

# MODEL DEVELOPMENT

In [29]:
# XGBoost (no scaling is needed for tree-based method)
from xgboost import XGBClassifier

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()

model = XGBClassifier(
    # n_estimators=400,#100,200,400,500
    # max_depth=4,#3-6
    # learning_rate=0.05,#0.03-0.1
    # subsample=0.8,#0.6-0.9
    # colsample_bytree=0.8,
    eval_metric="aucpr",
    scale_pos_weight=neg/pos,
    random_state=84,
    n_jobs=-1
)

# model.fit(x_train, y_train)
# y_pred_scr = model.predict_proba(x_test)[:, 1]

In [30]:
# Random Search for best hyperparameters
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=84
)

params = {
    'n_estimators': [100,200,400],
    'max_depth': [3,4,5],
    #'learning_rate': [0.03,0.05,0.1],
    #'subsample': [0.6,0.8,0.9]
}

search = RandomizedSearchCV(
    estimator=model,
    param_distributions=params,
    n_iter=5,
    scoring="average_precision",
    cv=skf,
    random_state=84,
    n_jobs=-1
)

search.fit(x_train, y_train)

print("Best PR-AUC:", search.best_score_)
print("Best params:", search.best_params_)

Best PR-AUC: 0.9997441383871891
Best params: {'n_estimators': 100, 'max_depth': 4}


In [31]:
# final model with the best parameters
final_model = search.best_estimator_
y_pred_scr = final_model.predict_proba(x_test)[:, 1]

In [32]:
# calculate PR_AUC for test data
from sklearn.metrics import precision_recall_curve, average_precision_score

precision, recall, thresholds = precision_recall_curve(
    y_test,
    y_pred_scr
)

pr_auc = average_precision_score(y_test, y_pred_scr)

print("PR-AUC:", pr_auc)

PR-AUC: 0.9997205014643301
